# Part 2 - Modelling and Simulations

Here we mix programmatic modelling/compositional modelling with advanced simulation workflows.


# Part 2.1 - Advanced Modelling

## Programmatic model construction (without the DSL)

We create a simple binding/dissociation model directly from symbolic components.


In [ ]:
using Catalyst, ModelingToolkit, OrdinaryDiffEq, StochasticDiffEq, DiffEqCallbacks, Plots

t = default_t()
@species X(t) Y(t) XY(t)
@parameters kB kD

reactions = [
    Reaction(kB, [X, Y], [XY], [1, 1], [1]),
    Reaction(kD, [XY], [X, Y], [1], [1, 1])
]

@named rs = ReactionSystem(reactions, t)
rs = complete(rs)

u0 = [X => 2.0, Y => 1.0, XY => 0.0]
ps = [kB => 1.0, kD => 1.0]
sol = solve(ODEProblem(rs, u0, (0.0, 10.0), ps), Tsit5())
plot(sol; title = "Programmatically created model")


Catalyst/ModelingToolkit models are symbolic, so we can manipulate expressions directly.


In [ ]:
eq = (kB^2 - kD^2) / (kB - kD)
simplify(eq)


## Coupling nutrient dynamics into the CRN

In older versions we often demonstrated compositional modelling with `extend`.
With the current setup, a robust workshop-friendly approach is to directly encode the nutrient ODE as a reaction law inside the CRN.

The depletion law `dN/dt = -r * P_a * N` is represented by the reaction `N --> 0` with rate `r*P_a*N`.


In [ ]:
rn_coupled = @reaction_network begin
    (mm(N, v, K), d), 0 <--> E
    (kA * E, kD), P_i <--> P_a
    r * P_a * N, N --> 0
end

u0 = [:N => 1.0, :E => 0.0, :P_i => 1.0, :P_a => 0.0]
ps = [:v => 1.0, :K => 0.2, :d => 0.3, :kA => 2.0, :kD => 0.5, :r => 0.2]
sol = solve(ODEProblem(rn_coupled, u0, (0.0, 15.0), ps), Tsit5())
plot(sol; title = "CRN with coupled nutrient depletion")


# Part 2.2 - Advanced Simulations

We next look at solution indexing, solver choices, ensemble simulations, and callbacks.


In [ ]:
brusselator = @reaction_network begin
    A, 0 --> X
    1, 2X + Y --> 3X
    B, X --> Y
    1, X --> 0
end

u0 = [:X => 1.0, :Y => 0.0]
ps = [:A => 1.0, :B => 4.0]
bruss_prob = ODEProblem(brusselator, u0, (0.0, 50.0), ps)
sol = solve(bruss_prob, Tsit5())

plot(sol; idxs = :X, title = "Brusselator X(t)")
plot(sol; idxs = (:X, :Y), title = "Brusselator phase portrait")


In [ ]:
sol_stiff = solve(bruss_prob, Rodas5P())
plot(sol_stiff; idxs = [:X, :Y], title = "Brusselator with Rodas5P")


### Monte Carlo simulations with `EnsembleProblem`

We simulate the same SDE multiple times.


In [ ]:
bistable_switch = @reaction_network begin
    v0 + hill(X, v, K, n), 0 --> X
    deg, X --> 0
end

sprob = SDEProblem(
    bistable_switch,
    [:X => 0.0],
    (0.0, 500.0),
    [:v0 => 0.1, :v => 2.5, :K => 75.0, :n => 2.0, :deg => 0.01],
)

eprob = EnsembleProblem(sprob)
esol = solve(eprob, ImplicitEM(), EnsembleThreads(); dt = 0.1, trajectories = 40)
plot(esol; linealpha = 0.35, title = "SDE ensemble")


### Callback example

At preset times, we add a pulse of `X` to a simple degradation model.


In [ ]:
degradation_model = @reaction_network begin
    d, X --> 0
end

oprob = ODEProblem(degradation_model, [:X => 10.0], (0.0, 10.0), [:d => 1.0])

condition = [3.0, 7.0]
affect!(integrator) = (integrator[:X] += 5.0)
pst_cb = PresetTimeCallback(condition, affect!)

sol_no_cb = solve(oprob, Tsit5())
sol_cb = solve(oprob, Tsit5(); callback = pst_cb)

plot(sol_no_cb; idxs = :X, label = "no callback", xlabel = "t", ylabel = "X")
plot!(sol_cb; idxs = :X, label = "with callback", title = "PresetTimeCallback example")
